In [140]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as gos
from plotly.subplots import make_subplots
import datetime


In [141]:
df = pd.read_csv('../data/shortage_clean.csv')
date_cols = ['initial_posting_date', 'update_date', 'discontinued_date']
df[date_cols] = df[date_cols].apply(pd.to_datetime, errors='coerce')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1686 entries, 0 to 1685
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   update_type           1686 non-null   object        
 1   initial_posting_date  1686 non-null   datetime64[ns]
 2   generic_name          1686 non-null   object        
 3   availability          1686 non-null   object        
 4   update_date           1686 non-null   datetime64[ns]
 5   therapeutic_category  1686 non-null   object        
 6   dosage_form           1675 non-null   object        
 7   company_name          1686 non-null   object        
 8   shortage_reason       1686 non-null   object        
 9   status                1686 non-null   object        
 10  discontinued_date     494 non-null    datetime64[ns]
 11  brand_name            1559 non-null   object        
 12  route                 1550 non-null   object        
 13  year              

In [142]:
df.head()

,update_type,initial_posting_date,generic_name,availability,update_date,therapeutic_category,dosage_form,company_name,shortage_reason,status,discontinued_date,brand_name,route,year,duration_days
0,Revised,2017-11-01,Hydromorphone Hydrochloride Injection,Unavailable,2026-02-25,Analgesia/Addiction,Injection,"Fresenius Kabi USA, LLC",Other,Current,NaT,DILAUDID,"INTRAMUSCULAR, INTRAVENOUS, SUBCUTANEOUS",2017,3038
1,New,2025-12-03,Sodium Chloride 0.9% Injection,To Be Discontinued,2025-12-03,Gastroenterology,Injection,"Hospira, Inc., a Pfizer Company",Not Applicable,To Be Discontinued,2025-12-03,SODIUM CHLORIDE,INTRAVENOUS,2025,0
2,New,2025-11-19,Hydrochlorothiazide Capsule,To Be Discontinued,2025-11-19,Cardiovascular,Capsule,"Teva Pharmaceuticals USA, Inc.",Not Applicable,To Be Discontinued,2025-11-19,HYDROCHLOROTHIAZIDE,ORAL,2025,0
3,Reverified,2018-03-27,Ropivacaine Hydrochloride Injection,Available,2025-11-13,Anesthesia,Injection,InfoRLife SA,Available,Current,NaT,ROPIVACAINE HYDROCHLORIDE,"EPIDURAL, INFILTRATION",2018,2788
4,Reverified,2022-10-12,"Amphetamine Aspartate Monohydrate, Amphetamine...",Limited Availability,2026-03-03,Psychiatry,Tablet,"Elite Laboratories, Inc.",Shortage of an active ingredient,Current,NaT,"DEXTROAMPHETAMINE SACCHARATE, AMPHETAMINE ASPA...",ORAL,2022,1238


## Temporal Pattern
Identifying trends in shortage over time.

In [143]:
def plot_temporal_trends(df, start_date='2010-01-01', end_date='2026-01-01', category = 'status'):
    
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    df_filtered = df[
        (df['initial_posting_date'] >= start) & 
        (df['initial_posting_date'] <= end)
    ]
    
    yearly = df_filtered.groupby('year').size().reset_index(name='count')
    yearly_status = df_filtered.groupby(['year', category]).size().reset_index(name='count')

    fig1 = px.line(yearly, x='year', y='count', 
                   title='Total Shortages Per Year', 
                   markers=True)
    
    fig2 = px.bar(yearly_status, x='year', y='count', 
                  color= category,
                  title=f'Shortages by {category} Per Year')
    
    fig1.show()
    fig2.show()

plot_temporal_trends(df, category='availability')


## Category Distribution
Distribution of therapeutic areas in which shortage has been recorded in the past

In [144]:
def bar_chart_categorical(df, status_category = 'availability', groupby_category = 'therapeutic_category'):
    category_counts = df[groupby_category].value_counts().reset_index().sort_values('count', ascending=False)
    category_counts = category_counts.head(10)
    category_counts.columns = [groupby_category, 'count']
    valid_categories = category_counts[groupby_category].tolist()

    category_with_availability = df[df[groupby_category].isin(valid_categories)]
    category_with_availability = category_with_availability.groupby([groupby_category, status_category]).size().reset_index(name='count').sort_values('count', ascending=False)

    fig1 = px.bar(category_counts, x=groupby_category, y='count', 
                 title=f'Shortage Counts by {groupby_category}',
                 labels={groupby_category: groupby_category.capitalize(), 'count': 'Shortage Count'})

    fig2 = px.bar(category_with_availability, x=groupby_category, y='count', color=status_category,
                  title=f'Shortages by {status_category.capitalize()} and {groupby_category.capitalize()}',
                  labels={groupby_category: groupby_category.capitalize(), 'count': 'Shortage Count', status_category: status_category.capitalize()})

    fig1.show()
    fig2.show()

bar_chart_categorical(df, status_category = 'availability', groupby_category = 'therapeutic_category')


# Reasons for unavailability:
Creating a pie chart to figure out the share of causes resulting in unavailability

In [145]:
def hbar_category_distribution(df):
    df = df[(df['availability'] != 'Available') & (df['shortage_reason'] !='Not Applicable')]  # Filter to only include unavailable drugs
    shortage_counts = df.groupby('shortage_reason').size().reset_index(name='count').sort_values('count', ascending=True)
    shortage_counts = shortage_counts[shortage_counts['count'] > 20]  # Filter out reasons with very low counts
    shortage_counts.columns = ['shortage_reason', 'count']
    fig = px.bar(shortage_counts, 
             x='count', 
             y='shortage_reason',
             orientation='h',
             title='Root Causes of Drug Unavailability',
             labels={'shortage_reason': 'Shortage Reason', 'count': 'Number of Shortages'})
    fig.show()
hbar_category_distribution(df)

In [146]:
# Generic code block for bar chart plotting for a categorical variable
def hbar_category_distribution(df, groupby_category = 'shortage_reason'):
    if groupby_category =='shortage_reason':
        df = df[(df['availability'] != 'Available') & (df['shortage_reason'] !='Not Applicable')]  # Filter to only include unavailable drugs
    
    shortage_counts = df.groupby(groupby_category).size().reset_index(name='count').sort_values('count', ascending=False)
    shortage_counts = shortage_counts[shortage_counts['count'] > np.median(shortage_counts['count'])].head(10)
    shortage_counts.columns = [groupby_category, 'count']
    fig = px.bar(shortage_counts, 
             x='count', 
             y=groupby_category,
             orientation='h',
             title=f'Drug shortage by {groupby_category}',
             labels={groupby_category: groupby_category.capitalize(), 'count': 'Number of Shortages'})
    fig.show()

hbar_category_distribution(df)

# Status Breakdown
Run df['status'].value_counts(). What percentage of shortages are currently active?



In [147]:
def pie_plot_category_distribution(df, groupby_category = 'availability'):
    shortage_counts = df.groupby(groupby_category).size().reset_index(name='count').sort_values('count', ascending=True)
    shortage_counts.columns = [groupby_category, 'count']
    # Pie chart for the  categorical variable:
    fig = px.pie(shortage_counts, values = 'count', names=groupby_category, title = f'Percentage of shortages distribution on {groupby_category}')
    fig.show()
pie_plot_category_distribution(df)


# Shortage by Dosage form

In [148]:
hbar_category_distribution(df,groupby_category = 'dosage_form')

# Shortage by Brand Name

In [149]:
hbar_category_distribution(df,groupby_category = 'brand_name')

# Shortage by Company Name

In [150]:
hbar_category_distribution(df,groupby_category = 'company_name')

In [ ]:
def unavailable_duration(df, category = 'availability'):
    reliable_states = ['Unavailable', 'To Be Discontinued', 'Resolved']
    df_duration = df[
        (df['duration_days'] >= 0) & 
        (df['availability'].isin(reliable_states))
    ]

    fig = px.box(df_duration, x=x_category, y='duration_days',
                title=f'Shortage Duration by {x_category}')
    fig.show()
unavailable_duration(df)

In [ ]:
unavailable_duration(df, category = 'therapeutic_category')

In [167]:
# Aggregate by category
def scatter_plot_durationvsshortage(df, category = 'therapeutic_category'):
    reliable_states = ['Unavailable', 'To Be Discontinued', 'Resolved']
    df_duration = df[
        (df['duration_days'] >= 0) & 
        (df['availability'].isin(reliable_states))
    ]
    scatter_data = df_duration.groupby(category).agg(
        shortage_count=('duration_days', 'count'),
        median_duration=('duration_days', 'median')
    ).reset_index()
    scatter_data = scatter_data.head(10)
    fig = px.scatter(scatter_data,
                    x='shortage_count',
                    y='median_duration',
                    text=category,
                    title='Shortage Frequency vs Duration by Therapeutic Category',
                    labels={'shortage_count': 'Number of Shortages',
                            'median_duration': 'Median Duration (days)'})
    fig.update_traces(textposition='top center')
    fig.show()

scatter_plot_durationvsshortage(df,category = 'company_name')